# LeetCode 1647 — Minimum Deletions to Make Character Frequencies Unique bằng Python

Link đề: https://leetcode.com/problems/minimum-deletions-to-make-character-frequencies-unique/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def minDeletions(self, s: str) -> int:
        ...
```

Notebook đi từ **vét cạn mọi tần suất đích** → tham lam với **tập tần suất đã dùng** → **sắp xếp giảm dần + trần `prev`** → **đếm theo "xô" tần suất** chạy tuyến tính.

## 1. Mục tiêu học

1. Chuyển bài về **mảng tần suất** (tối đa 26 số) — chỉ số lượng mỗi chữ quan trọng, không quan trọng vị trí.
2. Hiểu bài toán con: giảm (không được tăng) các số sao cho các số **dương** đôi một khác nhau, tổng lượng giảm nhỏ nhất; tần suất `0` được phép trùng.
3. Cài đặt tham lam bằng `set` các tần suất đã dùng, và bằng sắp xếp giảm dần với "trần" `prev − 1`.
4. Chứng minh tham lam bằng lập luận **hoán đổi / chặn dưới**, và biết phiên bản đếm xô `O(n)`.
5. Viết brute force trên mảng tần suất nhỏ để đối chiếu, test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Xâu `s` gọi là **tốt** nếu **không có hai ký tự khác nhau** nào có **cùng tần suất** (số lần xuất hiện). Cho `s`, trả về **số ký tự ít nhất** cần **xóa** để `s` trở thành tốt.

Tần suất của một ký tự là số lần nó xuất hiện. Ví dụ trong `"aab"`, tần suất của `'a'` là 2, của `'b'` là 1.

**Ví dụ 1:** `s = "aab"` → `0` (đã tốt).

**Ví dụ 2:** `s = "aaabbbcc"` → `2`. Xóa hai `'b'` được `"aaabcc"` (tần suất 3, 1, 2); hoặc xóa một `'b'` và một `'c'` được `"aaabbc"` (3, 2, 1).

**Ví dụ 3:** `s = "ceabaacb"` → `2`. Xóa cả hai `'c'` được `"eabaab"`. Chú ý: ta chỉ quan tâm các ký tự **còn lại** trong xâu (tần suất 0 bị bỏ qua).

**Ràng buộc:**
- `1 ≤ s.length ≤ 10⁵`
- `s` chỉ gồm chữ cái tiếng Anh in thường.

In [1]:
import random
import time
from collections import Counter
from itertools import product
from typing import List

examples = [("aab", 0), ("aaabbbcc", 2), ("ceabaacb", 2)]

def is_good(s):
    freqs = list(Counter(s).values())
    return len(freqs) == len(set(freqs))

for s, exp in examples:
    print(f"s={s!r}: tần suất={dict(Counter(s))}, tốt? {is_good(s)} -> kỳ vọng xóa {exp}")
print("Kiểm tra lời giải mẫu của ví dụ 2:", is_good("aaabcc"), is_good("aaabbc"))

s='aab': tần suất={'a': 2, 'b': 1}, tốt? True -> kỳ vọng xóa 0
s='aaabbbcc': tần suất={'a': 3, 'b': 3, 'c': 2}, tốt? False -> kỳ vọng xóa 2
s='ceabaacb': tần suất={'c': 2, 'e': 1, 'a': 3, 'b': 2}, tốt? False -> kỳ vọng xóa 2
Kiểm tra lời giải mẫu của ví dụ 2: True True


## 3. Kiến thức nền

### 3.1 `collections.Counter`

`Counter(s)` đếm số lần xuất hiện mỗi phần tử, trả về một `dict` con. `.values()` cho các tần suất, `.most_common()` sắp theo tần suất giảm dần.

In [2]:
c = Counter("ceabaacb")
print(c)
print("các tần suất:", list(c.values()))
print("most_common:", c.most_common())

Counter({'a': 3, 'c': 2, 'b': 2, 'e': 1})
các tần suất: [2, 1, 3, 2]
most_common: [('a', 3), ('c', 2), ('b', 2), ('e', 1)]


### 3.2 Sắp xếp giảm dần

`sorted(values, reverse=True)` cho list mới giảm dần; `list.sort(reverse=True)` sắp tại chỗ. Với tối đa 26 số, chi phí sắp xếp không đáng kể.

In [3]:
vals = [2, 3, 3, 1, 3]
print(sorted(vals, reverse=True), "| list gốc không đổi:", vals)
vals.sort(reverse=True)
print("sau sort tại chỗ:", vals)

[3, 3, 3, 2, 1] | list gốc không đổi: [2, 3, 3, 1, 3]
sau sort tại chỗ: [3, 3, 3, 2, 1]


### 3.3 `set` — kiểm tra thuộc tập trong `O(1)`

`x in used` kiểm tra nhanh; `used.add(x)` thêm phần tử. Thích hợp để ghi nhớ "tần suất nào đã có chủ".

In [4]:
used = set()
for f in [3, 3, 2]:
    print(f"f={f}: đã dùng? {f in used}")
    used.add(f)
print(used)

f=3: đã dùng? False
f=3: đã dùng? True
f=2: đã dùng? False
{2, 3}


### 3.4 Mảng đếm (counting / bucket)

Khi giá trị nằm trong `[0, n]`, mảng `bucket[v]` = "có bao nhiêu phần tử bằng `v`" thay cho sắp xếp, cho phép quét `O(n)`.

In [5]:
freqs = [3, 3, 3, 1]
bucket = [0] * (max(freqs) + 1)
for f in freqs:
    bucket[f] += 1
print("bucket[v] = số ký tự có tần suất v:", bucket)

bucket[v] = số ký tự có tần suất v: [0, 1, 0, 3]


## 4. Ý tưởng nền tảng / trực giác

Bài toán thực chất là trên **mảng tần suất** `F` (≤ 26 số dương): được **giảm** mỗi số (mỗi đơn vị giảm = xóa 1 ký tự), muốn các số **dương** còn lại đôi một khác nhau, tổng lượng giảm nhỏ nhất.

Tính tay `s = "aaabbbcc"` ⇒ `F = [3, 3, 2]`:
- Giữ `3` cho `a`.
- `b` cũng `3` — trùng ⇒ hạ xuống `2`? `2` đã thuộc về `c` ⇒ hạ tiếp xuống `1` ⇒ tốn 2.
- `c` giữ `2` ⇒ tổng 2.
Hoặc: `b` hạ xuống `2` (tốn 1), `c` hạ xuống `1` (tốn 1) ⇒ tổng 2. Hai cách cùng chi phí.

Tính tay `F = [4, 4, 4, 4, 1]` (ví dụ `"aaaabbbbccccdddde"`):
- Sắp giảm: `4, 4, 4, 4, 1`. Trần lần lượt: `4`, `3`, `2`, `1`, rồi `0`.
- Chi phí: `0 + 1 + 2 + 3 + 1 = 7` (chữ `e` có tần suất 1 phải xóa nốt vì `1` đã có chủ).

Quy luật: sau khi sắp **giảm dần**, mỗi số được giữ **tối đa** `min(f, giá_trị_giữ_trước − 1)`, không âm.

In [6]:
def manual(F):
    F = sorted(F, reverse=True)
    prev = float('inf')
    total = 0
    for f in F:
        keep = max(0, min(f, prev - 1))
        print(f"  f={f}: trần={prev - 1 if prev != float('inf') else '∞'}, giữ {keep}, xóa {f - keep}")
        total += f - keep
        prev = keep
    print("  tổng xóa:", total)
    return total

print("F = [3, 3, 2]"); manual([3, 3, 2])
print("F = [4, 4, 4, 4, 1]"); manual([4, 4, 4, 4, 1])

F = [3, 3, 2]
  f=3: trần=∞, giữ 3, xóa 0
  f=3: trần=2, giữ 2, xóa 1
  f=2: trần=1, giữ 1, xóa 1
  tổng xóa: 2
F = [4, 4, 4, 4, 1]
  f=4: trần=∞, giữ 4, xóa 0
  f=4: trần=3, giữ 3, xóa 1
  f=4: trần=2, giữ 2, xóa 2
  f=4: trần=1, giữ 1, xóa 3
  f=1: trần=0, giữ 0, xóa 1
  tổng xóa: 7


7

## 5. Cách 1 — Brute Force / baseline: thử mọi tần suất đích

**Ý tưởng:** với mảng tần suất `F`, thử **mọi** bộ đích `T` với `0 ≤ T[i] ≤ F[i]`; bộ hợp lệ nếu các `T[i] > 0` đôi một khác nhau; chi phí `Σ(F[i] − T[i])`. Lấy min. Dùng DFS có cắt nhánh "giá trị dương đã dùng".

In [7]:
def min_del_brute(s):
    F = list(Counter(s).values())
    best = float('inf')
    def dfs(i, used, cost):
        nonlocal best
        if cost >= best:
            return
        if i == len(F):
            best = cost
            return
        for t in range(F[i], -1, -1):              # thử giữ t ký tự
            if t == 0 or t not in used:
                if t: used.add(t)
                dfs(i + 1, used, cost + F[i] - t)
                if t: used.discard(t)
    dfs(0, set(), 0)
    return best

for s, exp in examples:
    print(s, "->", min_del_brute(s), "(kỳ vọng", exp, ")")

aab -> 0 (kỳ vọng 0 )
aaabbbcc -> 2 (kỳ vọng 2 )
ceabaacb -> 2 (kỳ vọng 2 )


**Độ phức tạp:** xấu nhất `Π (F[i] + 1)` bộ đích — với 26 chữ mỗi chữ tần suất `~4000` là con số khổng lồ. Chỉ dùng cho `s` ngắn (vài chục ký tự, ít chữ khác nhau). Cắt nhánh `cost >= best` giúp nhưng không đổi bản chất hàm mũ.

**Nút thắt:** ta thử cả những cách "rõ ràng tệ" (ví dụ xóa nhiều hơn cần). Một chữ chỉ nên giảm **vừa đủ** để tránh va chạm.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam:** xử lý từng chữ; với tần suất `f`, **nếu** `f` đã có chủ (và `f > 0`) **thì** giảm `f` đi 1 và thử lại; **nếu** `f` trống hoặc `f = 0` **thì** dừng, ghi nhận `f`.

Phiên bản sắp xếp: duyệt `F` **giảm dần**, giữ `prev` = giá trị vừa giữ của chữ trước. Chữ hiện tại giữ `keep = min(f, prev − 1)` (không âm). Lý do: các chữ đã duyệt chiếm các giá trị `> keep` khác nhau; chữ hiện tại không thể vượt `f`; cao nhất còn trống không vượt `prev − 1`.

**Bất biến (bản sắp xếp):** sau khi xử lý `k` số lớn nhất, các giá trị giữ được tạo thành dãy **giảm nghiêm ngặt** trên phần dương, `prev` là giá trị nhỏ nhất trong đó, và tổng xóa là **nhỏ nhất** cho `k` số này.

**Chặn dưới:** sắp `F` giảm dần `f₁ ≥ f₂ ≥ … ≥ f_m`. Trong mọi lời giải hợp lệ, giá trị giữ của số thứ `i` (theo thứ tự này) — sau khi sắp lại các giá trị giữ — không thể vượt `max(0, min(f_i, keep_{i−1} − 1))`. Vì vậy tham lam giữ được **nhiều nhất có thể** ⇒ xóa ít nhất (chứng minh chi tiết ở mục 10).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam với tập tần suất đã dùng

In [8]:
def min_del_set(s):
    used = set()
    ans = 0
    for f in Counter(s).values():
        while f > 0 and f in used:
            f -= 1
            ans += 1
        used.add(f)            # thêm cả 0 cũng không sao vì 0 không bao giờ bị chặn
    return ans

for s, exp in examples:
    print(s, "->", min_del_set(s), "(kỳ vọng", exp, ")")

aab -> 0 (kỳ vọng 0 )
aaabbbcc -> 2 (kỳ vọng 2 )
ceabaacb -> 2 (kỳ vọng 2 )


In [9]:
class SolutionSet:
    def minDeletions(self, s: str) -> int:
        used = set()
        ans = 0
        for f in Counter(s).values():
            while f > 0 and f in used:
                f -= 1
                ans += 1
            used.add(f)
        return ans

print([SolutionSet().minDeletions(s) for s, _ in examples])

[0, 2, 2]


**Độ phức tạp:** đếm `O(n)`; vòng `while` tổng cộng chạy đúng `ans` lần, mà `ans ≤ n` ⇒ `O(n)` thời gian; bộ nhớ `O(26)` cho `Counter` và `set`. (Đáng chú ý: thứ tự duyệt các tần suất **không ảnh hưởng** tới kết quả — bạn có thể tự kiểm bằng test ngẫu nhiên ở mục 12.)

### 7.2 Cách 3 — Sắp xếp giảm dần + vòng `while` hạ dần

Sắp giảm dần; với mỗi phần tử `vals[i]`, trong khi nó `≥ vals[i−1]` và còn dương, hạ đi 1.

In [10]:
def min_del_sort_while(s):
    vals = sorted(Counter(s).values(), reverse=True)
    ans = 0
    for i in range(1, len(vals)):
        while vals[i] >= vals[i - 1] and vals[i] > 0:
            vals[i] -= 1
            ans += 1
    return ans

for s, exp in examples:
    print(s, "->", min_del_sort_while(s), "(kỳ vọng", exp, ")")

aab -> 0 (kỳ vọng 0 )
aaabbbcc -> 2 (kỳ vọng 2 )
ceabaacb -> 2 (kỳ vọng 2 )


**Độ phức tạp:** sắp 26 số `O(1)`; vòng `while` tổng chạy `ans ≤ n` lần ⇒ `O(n)`.

### 7.3 Cách 4 — Sắp xếp giảm dần + trần `prev` (không vòng `while`)

Thay vì hạ từng đơn vị, tính thẳng `keep = min(f, prev − 1)` và cộng `f − keep`. Với `prev = 0`, mọi số còn lại phải xóa sạch.

In [11]:
def min_del_cap(s):
    ans = 0
    prev = float('inf')
    for f in sorted(Counter(s).values(), reverse=True):
        keep = max(0, min(f, prev - 1))
        ans += f - keep
        prev = keep
    return ans

for s, exp in examples:
    print(s, "->", min_del_cap(s), "(kỳ vọng", exp, ")")

aab -> 0 (kỳ vọng 0 )
aaabbbcc -> 2 (kỳ vọng 2 )
ceabaacb -> 2 (kỳ vọng 2 )


**Độ phức tạp:** `O(n)` để đếm + `O(26 log 26)` để sắp ⇒ `O(n)`; bộ nhớ `O(26)`. Đây là lời giải chính (mục 9).

### 7.4 Cách 5 — Đếm theo xô tần suất, quét từ cao xuống thấp

`bucket[v]` = số chữ có tần suất `v`. Quét `v` từ lớn xuống `1`, giữ `carry` = số chữ **đang "rơi" xuống** cần tìm chỗ:
- `carry += bucket[v]` (các chữ có tần suất đúng `v` gia nhập).
- Nếu `carry > 0`: một chữ chiếm mức `v` (`carry −= 1`); những chữ còn lại phải xuống `v − 1`, mỗi chữ tốn thêm 1 lần xóa: `ans += carry`.
Khi tới `v = 1`, các chữ còn trong `carry` bị đẩy xuống `0` — đã được tính phí ở bước cuối.

Cách này không cần sắp xếp, hữu ích khi số "loại" lớn (không chỉ 26 chữ).

In [12]:
def min_del_bucket(s):
    freqs = Counter(s).values()
    mx = max(freqs)
    bucket = [0] * (mx + 1)
    for f in freqs:
        bucket[f] += 1
    ans = carry = 0
    for v in range(mx, 0, -1):
        carry += bucket[v]
        if carry > 0:
            carry -= 1           # một chữ ở lại mức v
            ans += carry         # các chữ còn lại tụt xuống v-1
    return ans

for s, exp in examples:
    print(s, "->", min_del_bucket(s), "(kỳ vọng", exp, ")")

aab -> 0 (kỳ vọng 0 )
aaabbbcc -> 2 (kỳ vọng 2 )
ceabaacb -> 2 (kỳ vọng 2 )


**Độ phức tạp:** `O(n + max_freq)` = `O(n)` thời gian, `O(max_freq)` bộ nhớ.

## 8. TRACE từng bước

### 8.1 Trace cách 4 (trần `prev`) trên `s = "aaaabbbbccccdddde"` (ví dụ tự dựng)

Ví dụ tự dựng, đáp án được kiểm chứng bằng brute force bên dưới (nên đối chiếu lại đề gốc nếu muốn). Bảng mong đợi:

| f (giảm dần) | prev trước | keep = max(0, min(f, prev−1)) | xóa | ans |
|---|---|---|---|---|
| 4 | ∞ | 4 | 0 | 0 |
| 4 | 4 | 3 | 1 | 1 |
| 4 | 3 | 2 | 2 | 3 |
| 4 | 2 | 1 | 3 | 6 |
| 1 | 1 | 0 | 1 | 7 |

In [13]:
def trace_cap(s):
    freqs = sorted(Counter(s).items(), key=lambda kv: -kv[1])
    ans, prev = 0, float('inf')
    print(f"s={s!r}, tần suất giảm dần: {freqs}")
    for ch, f in freqs:
        keep = max(0, min(f, prev - 1))
        ans += f - keep
        pv = "∞" if prev == float('inf') else prev
        print(f"   '{ch}': f={f}, prev={pv}, keep={keep}, xóa {f - keep}, ans={ans}")
        prev = keep
    return ans

r = trace_cap("aaaabbbbccccdddde")
print("Kết quả:", r, "| brute force:", min_del_brute("aaaabbbbccccdddde"))

s='aaaabbbbccccdddde', tần suất giảm dần: [('a', 4), ('b', 4), ('c', 4), ('d', 4), ('e', 1)]
   'a': f=4, prev=∞, keep=4, xóa 0, ans=0
   'b': f=4, prev=4, keep=3, xóa 1, ans=1
   'c': f=4, prev=3, keep=2, xóa 2, ans=3
   'd': f=4, prev=2, keep=1, xóa 3, ans=6
   'e': f=1, prev=1, keep=0, xóa 1, ans=7
Kết quả: 7 | brute force: 7


### 8.2 Trace cách 2 (tập đã dùng) và cách 5 (xô) trên ví dụ 3 `"ceabaacb"`

In [14]:
def trace_set(s):
    used, ans = set(), 0
    for ch, f in Counter(s).items():
        start = f
        while f > 0 and f in used:
            f -= 1; ans += 1
        used.add(f)
        print(f"   '{ch}': f={start} -> giữ {f} (xóa {start - f}), used={sorted(used)}, ans={ans}")
    return ans

def trace_bucket(s):
    freqs = Counter(s).values()
    mx = max(freqs)
    bucket = [0] * (mx + 1)
    for f in freqs:
        bucket[f] += 1
    print("   bucket:", bucket)
    ans = carry = 0
    for v in range(mx, 0, -1):
        carry += bucket[v]
        stay = 1 if carry > 0 else 0
        carry -= stay
        ans += carry
        print(f"   v={v}: +{bucket[v]} chữ, {stay} chữ ở lại, {carry} chữ tụt xuống {v - 1}, ans={ans}")
    return ans

print("Cách 2:"); trace_set("ceabaacb")
print("Cách 5:"); trace_bucket("ceabaacb")

Cách 2:
   'c': f=2 -> giữ 2 (xóa 0), used=[2], ans=0
   'e': f=1 -> giữ 1 (xóa 0), used=[1, 2], ans=0
   'a': f=3 -> giữ 3 (xóa 0), used=[1, 2, 3], ans=0
   'b': f=2 -> giữ 0 (xóa 2), used=[0, 1, 2, 3], ans=2
Cách 5:
   bucket: [0, 1, 2, 1]
   v=3: +1 chữ, 1 chữ ở lại, 0 chữ tụt xuống 2, ans=0
   v=2: +2 chữ, 1 chữ ở lại, 1 chữ tụt xuống 1, ans=1
   v=1: +1 chữ, 1 chữ ở lại, 1 chữ tụt xuống 0, ans=2


2

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minDeletions(self, s: str) -> int:
        ans = 0
        prev = float('inf')                       # giá trị giữ của chữ trước (lớn hơn)
        for f in sorted(Counter(s).values(), reverse=True):
            keep = max(0, min(f, prev - 1))
            ans += f - keep
            prev = keep
        return ans

for s, exp in examples:
    assert Solution().minDeletions(s) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `f₁ ≥ f₂ ≥ … ≥ f_m` là tần suất đã sắp giảm dần và `k_i` là giá trị tham lam giữ: `k₁ = f₁`, `k_i = max(0, min(f_i, k_{i−1} − 1))`.

**Hợp lệ:** `k_i ≤ f_i` (chỉ xóa, không thêm), và phần dương của dãy `k` giảm nghiêm ngặt ⇒ các tần suất dương đôi một khác nhau.

**Tối ưu (chặn trên cho mọi lời giải):** xét một lời giải hợp lệ bất kỳ với giá trị giữ `t_i ≤ f_i`. Ta chứng minh `Σ t_i ≤ Σ k_i`. Sắp các `t_i` dương giảm dần: `u₁ > u₂ > … > u_r`. Ta có `u_j ≤ f_j` (số lớn thứ `j` trong `t` không vượt số lớn thứ `j` trong `f`, vì `t_i ≤ f_i` từng phần tử). Quy nạp: `u₁ ≤ f₁ = k₁`; nếu `u_{j−1} ≤ k_{j−1}` thì `u_j ≤ u_{j−1} − 1 ≤ k_{j−1} − 1` và `u_j ≤ f_j` ⇒ `u_j ≤ min(f_j, k_{j−1} − 1) ≤ k_j` (khi vế phải dương; nếu `k_j = 0` thì từ đó trở đi `u_j ≤ k_{j-1} - 1 ≤ 0`, mâu thuẫn với `u_j > 0`, nên `r < j`). Vậy `Σ t = Σ u ≤ Σ k` ⇒ tham lam **giữ nhiều nhất**, tức **xóa ít nhất**.

**Cách 2 (set):** mỗi chữ được đặt vào mức trống **cao nhất ≤ f** của nó; có thể chứng minh tổng mức giữ bằng `Σ k_i` bất kể thứ tự (một dạng "matroid" — tham lam lấy chỗ cao nhất). Ta kiểm chứng thực nghiệm bằng test với thứ tự ngẫu nhiên ở mục 12.

**Cách 5 (xô):** đúng là mô phỏng cách 4 theo từng mức `v`: tại mỗi mức chỉ một chữ được ở lại (giá trị khác nhau), các chữ còn lại tụt xuống — mỗi lần tụt 1 mức tốn 1 lần xóa.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`, `m ≤ 26` là số chữ khác nhau, `M` là tần suất lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(Π (f_i + 1))` | `O(m)` |
| 2. Set | `O(n + ans)` = `O(n)` | `O(m)` |
| 3. Sort + while | `O(n + m log m + ans)` = `O(n)` | `O(m)` |
| 4. Sort + trần | `O(n + m log m)` = `O(n)` | `O(m)` |
| 5. Xô | `O(n + M)` = `O(n)` | `O(M)` |

Mọi cách tối ưu đều bị chi phối bởi bước **đếm ký tự** `O(n)`. Đo trên `n = 10⁵`:

In [16]:
rnd = random.Random(1647)
big = ''.join(rnd.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5))
worst = ''.join(ch * 3846 for ch in "abcdefghijklmnopqrstuvwxyz")      # 26 chữ cùng tần suất
for name, fn in [("set", min_del_set), ("sort+while", min_del_sort_while), ("cap", min_del_cap), ("bucket", min_del_bucket)]:
    t0 = time.perf_counter(); a = fn(big); b = fn(worst); dt = time.perf_counter() - t0
    print(f"{name:<11}: ngẫu nhiên -> {a:>5}, 26 chữ bằng nhau -> {b:>5}   {dt * 1000:6.1f} ms")

set        : ngẫu nhiên ->     2, 26 chữ bằng nhau ->   325      4.4 ms
sort+while : ngẫu nhiên ->     2, 26 chữ bằng nhau ->   325      5.7 ms
cap        : ngẫu nhiên ->     2, 26 chữ bằng nhau ->   325      4.4 ms
bucket     : ngẫu nhiên ->     2, 26 chữ bằng nhau ->   325      4.8 ms


## 12. Bộ test

Đối chiếu với brute force trên xâu ngắn (ít chữ, tần suất nhỏ để brute force chạy nhanh), edge case, và kiểm tính chất "thứ tự duyệt không ảnh hưởng" của cách 2.

In [17]:
ALL = {
    "set": min_del_set,
    "sort_while": min_del_sort_while,
    "cap": min_del_cap,
    "bucket": min_del_bucket,
    "SolSet": lambda s: SolutionSet().minDeletions(s),
    "Solution": lambda s: Solution().minDeletions(s),
}

def check(s, exp=None, brute=True):
    ref = min_del_brute(s) if brute else min_del_cap(s)
    for name, fn in ALL.items():
        assert fn(s) == ref, (name, s, fn(s), ref)
    if exp is not None:
        assert ref == exp, (s, ref, exp)
    return ref

for s, exp in examples:
    check(s, exp)
print("Ví dụ đề: OK")

check("a", 0)
check("abc", 2)                    # 1,1,1 -> 1,0,0
check("aabbcc", 3)                 # 2,2,2 -> 2,1,0
check("aabbccdd", 5)               # 2,2,2,2 -> 2,1,0,0
check("aaabbbccc", 3)              # 3,3,3 -> 3,2,1
check("abcdefghijklmnopqrstuvwxyz", 25)
check("bbcebab", 2)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [18]:
rnd = random.Random(2022)
for _ in range(2500):
    k = rnd.randint(1, 5)
    letters = "abcde"[:k]
    s = ''.join(rnd.choice(letters) for _ in range(rnd.randint(1, 14)))
    check(s)
print("2500 xâu ngẫu nhiên nhỏ so với brute force: OK")

# Cách 2 với thứ tự duyệt ngẫu nhiên cho cùng kết quả
def min_del_set_order(freqs):
    used, ans = set(), 0
    for f in freqs:
        while f > 0 and f in used:
            f -= 1; ans += 1
        used.add(f)
    return ans

for _ in range(2000):
    F = [rnd.randint(1, 30) for _ in range(rnd.randint(1, 26))]
    base = min_del_set_order(sorted(F, reverse=True))
    for _ in range(3):
        rnd.shuffle(F)
        assert min_del_set_order(F) == base
print("Thứ tự duyệt không ảnh hưởng (2000 mảng tần suất): OK")

for _ in range(30):
    s = ''.join(rnd.choice("abcdefghijklmnopqrstuvwxyz"[:rnd.randint(1, 26)]) for _ in range(rnd.randint(1, 3000)))
    check(s, brute=False)
print("30 xâu cỡ lớn so giữa các cách: OK")

2500 xâu ngẫu nhiên nhỏ so với brute force: OK
Thứ tự duyệt không ảnh hưởng (2000 mảng tần suất): OK
30 xâu cỡ lớn so giữa các cách: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm **số chữ bị trùng** thay vì **số ký tự phải xóa**

In [19]:
def min_del_bug_count_dups(s):
    freqs = list(Counter(s).values())
    return len(freqs) - len(set(freqs))       # SAI: chỉ đếm số chữ bị trùng tần suất

for s in ["aaabbbccc", "aabbccdd"]:
    print(f"{s}: sai = {min_del_bug_count_dups(s)}, đúng = {Solution().minDeletions(s)}")

aaabbbccc: sai = 2, đúng = 3
aabbccdd: sai = 3, đúng = 5


**Vì sao sai:** một chữ bị trùng có thể phải hạ **nhiều** đơn vị (`3,3,3` → `3,2,1` tốn `0 + 1 + 2 = 3`), và việc hạ có thể gây **trùng dây chuyền**. **Sửa:** mô phỏng việc hạ (set) hoặc dùng trần `prev`.

### Lỗi 2 — Công thức trần quên trường hợp `prev = 0`

In [20]:
def min_del_bug_cap(s):
    ans = 0
    prev = float('inf')
    for f in sorted(Counter(s).values(), reverse=True):
        if f >= prev:
            ans += f - prev + 1               # SAI khi prev = 0: xóa f + 1 ký tự (nhiều hơn số đang có)
            prev -= 1                         # và prev thành âm
        else:
            prev = f
    return ans

s = "aabbccdd"
print("sai :", min_del_bug_cap(s), "| đúng:", Solution().minDeletions(s))

sai : 6 | đúng: 5


**Vì sao sai:** với tần suất `2,2,2,2`: sau `2, 1, 0` thì `prev = 0`; chữ thứ tư bị tính xóa `2 − 0 + 1 = 3` (chỉ có 2 ký tự!). **Sửa:** khi `prev == 0` thì xóa đúng `f` (giữ `0`), hoặc dùng `keep = max(0, min(f, prev − 1))`.

### Lỗi 3 — Coi tần suất `0` cũng phải duy nhất (vòng lặp vô hạn / xóa thừa)

In [21]:
def min_del_bug_zero(s, max_steps=1000):
    used, ans = set(), 0
    for f in Counter(s).values():
        steps = 0
        while f in used and steps < max_steps:   # SAI: không có điều kiện f > 0
            f -= 1; ans += 1; steps += 1         # (giới hạn steps chỉ để demo không treo)
        used.add(f)
    return ans

s = "abc"
print("sai :", min_del_bug_zero(s), "| đúng:", Solution().minDeletions(s))

sai : 3 | đúng: 2


**Vì sao sai:** với `"abc"` (1,1,1): chữ thứ hai hạ xuống `0`, chữ thứ ba hạ xuống `0` thấy `0` đã dùng nên hạ tiếp xuống `−1` — xóa nhiều hơn số ký tự có! (Ở đây đếm ra 3 thay vì 2.) Trong trường hợp khác có thể lặp vô hạn. Đề nói rõ: tần suất `0` bị bỏ qua. **Sửa:** `while f > 0 and f in used`.

### Lỗi 4 — Sắp xếp **khóa** thay vì **giá trị**

In [22]:
def min_del_bug_keys(s):
    ans, prev = 0, float('inf')
    cnt = Counter(s)
    for ch in sorted(cnt, reverse=True):          # SAI: sắp theo chữ cái, không theo tần suất
        f = cnt[ch]
        keep = max(0, min(f, prev - 1))
        ans += f - keep
        prev = keep
    return ans

s = "abbbcc"
print("sai :", min_del_bug_keys(s), "| đúng:", Solution().minDeletions(s))

sai : 3 | đúng: 0


**Vì sao sai:** công thức trần `prev − 1` **chỉ đúng khi duyệt tần suất giảm dần**. Sắp theo chữ cái (`c, b, a` với tần suất `2, 3, 1`) làm `b` bị ép xuống `1` dù nó lớn nhất. **Sửa:** `sorted(cnt.values(), reverse=True)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề hỏi về **tần suất** ký tự ⇒ bước đầu luôn là `Counter`, sau đó quên xâu đi, chỉ làm trên ≤ 26 số.
- "Làm các giá trị **khác nhau** với chi phí nhỏ nhất, chỉ được **giảm**" ⇒ sắp **giảm dần**, mỗi số lấy `min(chính nó, số trước − 1)`.
- "Chỉ được **tăng**" (LeetCode 945 — Minimum Increment to Make Array Unique) ⇒ sắp **tăng dần**, mỗi số lấy `max(chính nó, số trước + 1)`.
- Giá trị `0` thường là "thoát" — đọc kỹ đề xem `0` có cần duy nhất không.
- Khi miền giá trị nhỏ (≤ n), có thể thay sắp xếp bằng **mảng xô**.

## 15. Biến thể của bài

1. **LeetCode 945 — Minimum Increment to Make Array Unique:** chỉ được tăng ⇒ sắp tăng dần, đẩy lên.
2. **Được cả tăng (thêm ký tự) lẫn giảm**, mỗi thao tác chi phí 1: bài khó hơn, DP/ghép cặp theo mức.
3. **Chi phí xóa khác nhau theo chữ:** chữ "đắt" nên được giữ mức cao ⇒ tham lam theo thứ tự ưu tiên có thể không còn đúng — cần DP/luồng.
4. **Đếm số xâu con "tốt"** hoặc **xâu con dài nhất là tốt**: kết hợp cửa sổ trượt + kiểm tra tần suất.
5. **Mảng số nguyên bất kỳ** (không chỉ 26 loại): cách xô `O(n + M)` hoặc sắp xếp `O(m log m)`.

In [23]:
def min_increment_unique(nums):
    """LeetCode 945: chỉ được tăng, làm mọi phần tử khác nhau."""
    ans, need = 0, float('-inf')
    for x in sorted(nums):
        target = max(x, need)
        ans += target - x
        need = target + 1
    return ans

print("945 [3,2,1,2,1,7] ->", min_increment_unique([3, 2, 1, 2, 1, 7]))
print("945 [1,2,2] ->", min_increment_unique([1, 2, 2]))

945 [3,2,1,2,1,7] -> 6
945 [1,2,2] -> 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi tần suất đích | hàm mũ | `O(m)` | đối chiếu |
| 2. Set | hạ tới khi gặp mức trống | `O(n)` | `O(m)` | dễ viết nhất, không cần sắp |
| 3. Sort + while | hạ từng đơn vị theo số trước | `O(n)` | `O(m)` | trực quan |
| 4. Sort + trần | `keep = min(f, prev−1)` | `O(n)` | `O(m)` | **nộp bài**, dễ chứng minh |
| 5. Xô | quét mức từ cao xuống, `carry` | `O(n + M)` | `O(M)` | nhiều loại phần tử, tránh sắp xếp |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minDeletions("aaaabbbbcc")` bằng bao nhiêu?

**Bài 2 (tính tay):** Với mảng tần suất `[5, 5, 5, 2, 2, 1]`, tính bằng tay theo cách 4 (bảng `f`, `prev`, `keep`, `xóa`).

**Bài 3 (điền TODO):** Hoàn thiện `final_freqs(s)` trả về `dict` chữ → tần suất **sau khi xóa** theo phương án tham lam (cách 4), và tự kiểm tra kết quả hợp lệ.

**Bài 4 (tự viết lại):** Viết lại cách 5 (xô) cho **mảng số nguyên tùy ý** `freqs` (không cần xâu), đặt tên `min_dec_unique(freqs)`.

**Bài 5 (mini challenge):** Biến thể có trọng số: xóa một ký tự chữ `c` tốn `w[c]`. Viết `min_cost_weighted(s, w)` bằng brute force (vét cạn tần suất đích) cho xâu ngắn, và cho thấy tham lam cách 4 (bỏ qua trọng số) có thể **không** tối ưu.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def final_freqs(s):
    # TODO: sắp các (chữ, tần suất) giảm dần theo tần suất, áp dụng keep = max(0, min(f, prev - 1))
    pass

# Bài 4 — TODO
def min_dec_unique(freqs):
    # TODO: bucket[v] = số phần tử bằng v; quét v từ max xuống 1 với carry
    pass

# Bài 5 — TODO
def min_cost_weighted(s, w):
    # TODO: DFS thử mọi tần suất đích như min_del_brute, chi phí (f - t) * w[ch]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Tần suất `4, 4, 2`.
- Mức 2: Chữ thứ hai giữ `3`.
- Mức 3: Chữ thứ ba giữ `2` (≤ 3 − 1) ⇒ không phải xóa.

**Bài 2**
- Mức 1: Trần lần lượt `5, 4, 3, …`.
- Mức 2: `keep = 5, 4, 3, 2, 1, 0`.
- Mức 3: Xóa `0 + 1 + 2 + 0 + 1 + 1 = 5`.

**Bài 3**
- Mức 1: Dùng `Counter(s).items()` và sắp theo `-f`.
- Mức 2: Có thể bỏ những chữ có `keep = 0` khỏi `dict`.
- Mức 3: Kiểm tra: mọi `keep ≤ f`, các `keep > 0` khác nhau, tổng `f − keep` bằng `Solution().minDeletions(s)`.

**Bài 4**
- Mức 1: Giống hệt `min_del_bucket` nhưng nhận `freqs` trực tiếp.
- Mức 2: Bỏ qua phần tử `0` (đã hợp lệ).
- Mức 3: Kiểm với `min_del_cap` bằng cách dựng xâu từ `freqs`.

**Bài 5**
- Mức 1: Brute force: giống `min_del_brute` nhưng nhân trọng số.
- Mức 2: Tham lam chọn chữ nào "giữ mức cao" chỉ dựa trên tần suất.
- Mức 3: Thử `s = "aabb"`, `w = {'a': 1, 'b': 100}`: nên hạ `a` chứ không phải `b`. Tìm thêm phản ví dụ bằng tìm kiếm ngẫu nhiên.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
print("Bài 1:", Solution().minDeletions("aaaabbbbcc"))
assert Solution().minDeletions("aaaabbbbcc") == 1 == min_del_brute("aaaabbbbcc")

# Bài 2
s2 = "aaaaa" + "bbbbb" + "ccccc" + "dd" + "ee" + "f"
print("Bài 2:"); r2 = manual([5, 5, 5, 2, 2, 1])
assert r2 == 5 == Solution().minDeletions(s2)

# Bài 3
def final_freqs_sol(s):
    res, prev = {}, float('inf')
    for ch, f in sorted(Counter(s).items(), key=lambda kv: -kv[1]):
        keep = max(0, min(f, prev - 1))
        if keep:
            res[ch] = keep
        prev = keep
    return res

rnd = random.Random(3)
for _ in range(1000):
    s = ''.join(rnd.choice("abcdef") for _ in range(rnd.randint(1, 40)))
    ff, cnt = final_freqs_sol(s), Counter(s)
    assert all(ff[c] <= cnt[c] for c in ff)
    assert len(set(ff.values())) == len(ff)
    assert sum(cnt.values()) - sum(ff.values()) == Solution().minDeletions(s)
print("Bài 3 OK | ví dụ:", final_freqs_sol("aaabbbcc"))

Bài 1: 1
Bài 2:
  f=5: trần=∞, giữ 5, xóa 0
  f=5: trần=4, giữ 4, xóa 1
  f=5: trần=3, giữ 3, xóa 2
  f=2: trần=2, giữ 2, xóa 0
  f=2: trần=1, giữ 1, xóa 1
  f=1: trần=0, giữ 0, xóa 1
  tổng xóa: 5
Bài 3 OK | ví dụ: {'a': 3, 'b': 2, 'c': 1}


In [26]:
# Bài 4
def min_dec_unique_sol(freqs):
    freqs = [f for f in freqs if f > 0]
    if not freqs:
        return 0
    mx = max(freqs)
    bucket = [0] * (mx + 1)
    for f in freqs:
        bucket[f] += 1
    ans = carry = 0
    for v in range(mx, 0, -1):
        carry += bucket[v]
        if carry:
            carry -= 1
            ans += carry
    return ans

rnd = random.Random(4)
for _ in range(2000):
    F = [rnd.randint(0, 20) for _ in range(rnd.randint(1, 26))]
    s = ''.join(chr(97 + i) * f for i, f in enumerate(F))
    assert min_dec_unique_sol(F) == (Solution().minDeletions(s) if s else 0)
print("Bài 4 OK")

# Bài 5
def min_cost_weighted_sol(s, w):
    items = list(Counter(s).items())
    best = float('inf')
    def dfs(i, used, cost):
        nonlocal best
        if cost >= best:
            return
        if i == len(items):
            best = cost; return
        ch, f = items[i]
        for t in range(f, -1, -1):
            if t == 0 or t not in used:
                if t: used.add(t)
                dfs(i + 1, used, cost + (f - t) * w[ch])
                if t: used.discard(t)
    dfs(0, set(), 0)
    return best

def greedy_ignoring_weights(s, w):
    cost, prev = 0, float('inf')
    for ch, f in sorted(Counter(s).items(), key=lambda kv: -kv[1]):
        keep = max(0, min(f, prev - 1))
        cost += (f - keep) * w[ch]; prev = keep
    return cost

w = {'a': 1, 'b': 100}
print("Bài 5: aabb tối ưu =", min_cost_weighted_sol("aabb", w), "| tham lam bỏ qua trọng số =", greedy_ignoring_weights("aabb", w))
rnd = random.Random(6)
found = None
for _ in range(5000):
    s = ''.join(rnd.choice("abc") for _ in range(rnd.randint(2, 9)))
    w = {c: rnd.randint(1, 5) for c in "abc"}
    if greedy_ignoring_weights(s, w) != min_cost_weighted_sol(s, w):
        found = (s, w); break
print("Phản ví dụ ngẫu nhiên:", found, "->", greedy_ignoring_weights(*found), "vs", min_cost_weighted_sol(*found))
assert min_cost_weighted_sol("aabb", {'a': 1, 'b': 100}) == 1
assert all(min_cost_weighted_sol(s, {c: 1 for c in s}) == Solution().minDeletions(s) for s, _ in examples)

Bài 4 OK
Bài 5: aabb tối ưu = 1 | tham lam bỏ qua trọng số = 100
Phản ví dụ ngẫu nhiên: ('cbbabbacb', {'a': 5, 'b': 5, 'c': 1}) -> 5 vs 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "chỉ được giảm, làm các giá trị dương khác nhau":**

```python
ans, prev = 0, float('inf')
for f in sorted(values, reverse=True):
    keep = max(0, min(f, prev - 1))
    ans += f - keep
    prev = keep
return ans
```

**Template set:** `while f > 0 and f in used: f -= 1; ans += 1` rồi `used.add(f)`.

**Template xô:** `carry += bucket[v]; if carry: carry -= 1; ans += carry` với `v` từ `max` xuống `1`.

**Checklist:**
- [ ] Làm trên **tần suất** (`Counter(s).values()`), không phải trên khóa?
- [ ] Sắp **giảm dần** khi dùng trần `prev − 1`?
- [ ] Tần suất `0` được phép trùng (điều kiện `f > 0`)?
- [ ] Không xóa quá số ký tự đang có (`keep ≥ 0`)?
- [ ] Đã so với brute force trên xâu ngắn ngẫu nhiên?
- [ ] Chữ ký đúng: `def minDeletions(self, s: str) -> int`?